In [7]:
!pip install mistralai gradio

## Mistral Agent with JSON-Schema


1.   [Custom Structured Outputs](https://docs.mistral.ai/capabilities/structured_output/custom)
2.   [Mistral AI Studio](https://console.mistral.ai/home)



In [8]:
import os

from mistralai.client import Mistral

client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY"))

inputs = [
        {
            "role": "user",
            "content": "Mi a mai legfontosabb hírek?"
        }
    ]

completion_args = {
    "temperature": 0.45,
    "max_tokens": 4096,
    "top_p": 0.2,
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "response_schema",
            "schema": {
                "type": "object",
                "title": "StructuredData",
                "required": [
                    "news_title",
                    "news_content",
                    "news_topics",
                    "news_url"
                ],
                "properties": {
                    "news_url": {
                        "type": "string",
                        "format": "uri",
                        "description": "news' URL"
                    },
                    "news_title": {
                        "type": "string",
                        "description": "News' title"
                    },
                    "news_topics": {
                        "type": "string",
                        "description": "News' topics"
                    },
                    "news_content": {
                        "type": "string",
                        "description": "News' content"
                    }
                }
            }
        }
    },
}

tools = [
    {
        "toolConfiguration": None,
        "type": "web_search"
    }
]

response = client.beta.conversations.start(
    inputs=inputs,
    model="mistral-medium-latest",
    #instructions="Híradós szerkesztő vagy. Feladatod az, hogy az aktuális napi informatikai, technológiai hírek anyagját szerkezd meg, foglald össze magyarul, majd küldd el a hírcsatornáknak. Tárgyilagos és fókuszált legyél. Válaszaidban több helyről szerezd be az információkat. Híreket csoportokba szedve küldd el folyó szövegként.",
    instructions="you are J. Jonah Jameson, the loud, impatient, hard-edged editor-in-chief of the Daily Bugle. You speak in short, firm, commanding sentences. You believe in law and order, and you hate masked self-styled heroes — especially Spider-Man, who you think is a menace. You are always assertive, confrontational, and sarcastic. You never praise Spider-Man. Stay in character in every response.",
    completion_args=completion_args,
    tools=tools,
)

print(response)

SDKError: API error occurred: Status 401 Content-Type "application/json; charset=utf-8". Body: {"detail":"Unauthorized"}

## Mistral Agent SDK + Pydantic AI + Tools hasznáhasznáaa

In [9]:
import os
from mistralai.client import Mistral
from google.colab import userdata

os.environ['MISTRAL_API_KEY'] = userdata.get('MISTRAL_API_KEY')
client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY"))

inputs = [
    {
        "role": "user",
        "content": "Gyűjtsd össze a mai legfontosabb híreket külön-külön kifejtve!"
    }
]

completion_args = {
    "temperature": 0.45,
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "news_list_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "news_items": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "required": ["news_title", "news_content", "news_url", "news_topics"],
                            "properties": {
                                "news_title": {"type": "string", "description": "News' title"},
                                "news_content": {"type": "string", "description": "News' content"},
                                "news_url": {"type": "string", "format": "uri", "description": "news' URL"},
                                "news_topics": {"type": "string", "description": "News' topics"}
                            }
                        }
                    }
                },
                "required": ["news_items"]
            }
        }
    }
}

tools = [{"type": "web_search"}]

response = client.beta.conversations.start(
    inputs=inputs,
    model="mistral-medium-latest",
    instructions="Gyors, trendérzékeny, lényegretörő híradós hírszerkeztő vagy természetvédelem témakörben. A hírből azonnali, platformspecifikus posztot gyártasz Facebookra. A híreket egy 'news_items' nevű listába rendezd a megadott séma szerint.",
    completion_args=completion_args,
    tools=tools
)

In [10]:
import json

for entry in response.outputs:
    if entry.type == 'message.output':
        try:
            data = json.loads(entry.content)
            print(f"--- {len(data['news_items'])} hír érkezett ---\n")

            for i, item in enumerate(data['news_items'], 1):
                print(f"{i}. {item['news_title'].upper()}")
                print(f"Témák: {item['news_topics']}")
                print(f"Forrás: {item['news_url']}")
                print(f"{item['news_content']}\n" + "-"*30 + "\n")

        except (json.JSONDecodeError, KeyError) as e:
            print(f"Hiba a feldolgozás során: {e}")

--- 5 hír érkezett ---

1. AZ EURÁZSIAI HÓDOK VISSZATÉRÉSE
Témák: természetvédelem, klímaváltozás, eurázsiai hódok
Forrás: https://www.portfolio.hu/gazdasag/20260323/varatlan-eredmenyt-hozott-a-friss-kutatas-egy-ragcsalo-lehet-az-emberiseg-titkos-fegyvere-a-klimaharcban-825988
Az utóbbi években a természetvédelmi intézkedéseknek köszönhetően egyre nagyobb területekre térnek vissza az eurázsiai hódok, amelyek korábban majdhogynem teljesen eltűntek Európából a vadászat miatt. Tevékenységük jelentős hatást gyakorol a tájra.
------------------------------

2. MADÁRGYŰRŰZÉS ÉS NEMZETKÖZI EGYÜTTMŰKÖDÉS
Témák: természetvédelem, madárvédelem, nemzetközi együttműködés
Forrás: https://greenfo.hu/hirek/termeszetvedelmi-hirek/
A madárgyűrűzés alkalmazása csak nemzetközi összefogással lehetséges, amely mind a határon túli együttműködés, mind a tudományos eredmények révén segíti a vonuló madarak védelmét. Balczó Bertalan, az Agrárminisztérium természetvédelemért felelős helyettes államtitkára ezt ha

[Gradio UI framework](https://www.gradio.app/guides/quickstart)

In [12]:
import gradio as gr
import json

def get_tech_news():
    # Felugró tájékoztató ablak megjelenítése
    gr.Info("A hírek lekérése folyamatban! Néhány percet vesz igénybe.")

    try:
        # A korábbi cellában definiált logika meghívása
        inputs = [
            {
                "role": "user",
                "content": "Gyűjtsd össze a mai legfontosabb híreket külön-külön kifejtve!"
            }
        ]

        response = client.beta.conversations.start(
            inputs=inputs,
            model="mistral-medium-latest",
            instructions=" Szigorú, szkeptikus, adatorientált hírszerkeztő vagy természetvédelem témakörben. Nem írsz homályos megfogalmazásokat. A híreket egy 'news_items' nevű listába rendezd a megadott séma szerint.",
            completion_args=completion_args,
            tools=tools
        )

        output_html = ""
        for entry in response.outputs:
            if entry.type == 'message.output':
                data = json.loads(entry.content)
                for item in data['news_items']:
                    output_html += f"<div style='border: 1px solid #ddd; padding: 10px; margin-bottom: 10px; border-radius: 5px;'>"
                    output_html += f"<h3>{item['news_title']}</h3>"
                    output_html += f"<p><b>Témák:</b> {item['news_topics']}</p>"
                    output_html += f"<p>{item['news_content']}</p>"
                    output_html += f"<a href='{item['news_url']}' target='_blank'>Forrás megnyitása</a>"
                    output_html += "</div>"
        return output_html
    except Exception as e:
        return f"Hiba történt: {str(e)}"

with gr.Blocks() as demo:
    gr.Markdown("# 📰 Technológiai Híradó")
    gr.Markdown("Kattints a gombra a legfrissebb hírek letöltéséhez a Mistral AI segítségével.")
    btn = gr.Button("Hírek lekérése")
    output = gr.HTML(label="Hírek")

    btn.click(fn=get_tech_news, outputs=output)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://884416353f93e47ed7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 